# Baseline v2 — DINO cascade pancréas→tumeur + MedSAM

Reproduction isolée de `test_msd_no_resnet.py` (collaborateur), montée par-dessus le stack `msd_implementation/pipelines/dino_medsam_gemini/` existant.

**Étapes 1 à 4 du plan de priorité** :
1. P0.1 — vérifier que DINO pointe bien sur `best_coco_bbox_mAP_epoch_25.pth`
2. P0.2 — prompt `"pancreas . tumor ."` avec **filtrage par label** (pas de revert prompt)
3. P0.3 — seuil tumeur abaissé à `0.01` (seuil pancréas à `0.3`) — override runtime, pas de modif du repo principal
4. Rerun du baseline sur 100 images de test et comparaison au baseline précédent

## 0. Bootstrap Colab (clone + deps + symlinks Drive)

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'clean-structure',
            'https://github.com/moradBMH/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run — data/ et work_dirs/ doivent déjà être peuplés.')

## 1. Imports & vérification P0.1 (checkpoint epoch 25)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from msd_implementation.pipelines.dino_medsam_gemini import config
from colab.drive_paths import output_dir
from msd_implementation.pipelines.dino_medsam_cascade import cascade_detector as dino_v2, evaluation as eval_v2

assert config.DINO_CHECKPOINT.exists(), f'DINO checkpoint introuvable : {config.DINO_CHECKPOINT}'
assert 'epoch_25' in config.DINO_CHECKPOINT.name, (
    f'Le checkpoint actif est {config.DINO_CHECKPOINT.name}. Attendu : best_coco_bbox_mAP_epoch_25.pth'
)

print('CUDA     :', torch.cuda.is_available())
print('DINO cfg :', config.DINO_CONFIG)
print('DINO ckpt:', config.DINO_CHECKPOINT.name, '(P0.1 ✓ epoch 25)')
print('MedSAM   :', config.MEDSAM_CHECKPOINT.name)
print('MSD test :', config.MSD_TEST_JSON)

## 2. Overrides P0.2 + P0.3 (runtime, sans toucher `config.py` principal)

- **P0.2** — prompt `"pancreas . tumor ."` mais on lit `instances.labels` pour séparer pancréas (label 0) de tumeur (label 1). Voir [`dino_v2.py`](dino_v2.py) :
  - `detect_with_labels` retourne `[LabeledBox(xyxy, score, label), …]`
  - `cascade_select_tumor_box` applique la cascade du collaborateur : meilleure box pancréas (`score>0.3`), marge +20 px, tumeur doit chevaucher ≥ 10 %, fallback = meilleure tumeur si pas de pancréas.
- **P0.3** — seuils par classe : `pancreas_thr=0.3`, `tumor_thr=0.01`.

In [ ]:
TEXT_PROMPT       = 'pancreas . tumor .'
PANCREAS_THR      = 0.3
TUMOR_THR         = 0.01
PANCREAS_MARGIN_PX = 20
MIN_OVERLAP_RATIO  = 0.1

print(f"Prompt            : '{TEXT_PROMPT}'")
print(f"Seuil pancréas     : {PANCREAS_THR}  (label 0)")
print(f"Seuil tumeur       : {TUMOR_THR}  (label 1)")
print(f"Marge box pancréas : +{PANCREAS_MARGIN_PX} px")
print(f"Overlap min tumor/pancréas : {MIN_OVERLAP_RATIO}")

### 2.1 Sanity check — une image de test

Vérifier sur un seul scan que le filtrage par label fonctionne et que la cascade renvoie bien une box tumeur.

In [ ]:
with open(config.MSD_TEST_JSON) as f:
    test_meta = json.load(f)

sample = test_meta['images'][0]
sample_path = config.MSD_ROOT / sample['file_name']
print('Image de test :', sample_path)

raw = dino_v2.detect_with_labels(str(sample_path), text=TEXT_PROMPT, score_threshold=min(PANCREAS_THR, TUMOR_THR))
print(f"Nombre de boxes brutes : {len(raw)}")
for b in raw[:8]:
    cls = 'pancreas' if b.label == dino_v2.PANCREAS_LABEL else 'tumor'
    print(f"  label={b.label}({cls})  score={b.score:.3f}  xyxy={tuple(round(v,1) for v in b.xyxy)}")

selected = dino_v2.cascade_select_tumor_box(
    str(sample_path), text=TEXT_PROMPT,
    pancreas_thr=PANCREAS_THR, tumor_thr=TUMOR_THR,
    pancreas_margin_px=PANCREAS_MARGIN_PX, min_overlap_ratio=MIN_OVERLAP_RATIO,
)
print('\nCascade → box tumeur sélectionnée :', selected)

## 3. Rerun baseline complet (P0.4)

100 images de test. Attendu ~5-10 min sur T4.

In [ ]:
N_TEST = None  # None = les 100 images du split test

df_v2, summ_v2 = eval_v2.run_baseline_v2(
    coco_json=config.MSD_TEST_JSON,
    text=TEXT_PROMPT,
    pancreas_thr=PANCREAS_THR,
    tumor_thr=TUMOR_THR,
    pancreas_margin_px=PANCREAS_MARGIN_PX,
    min_overlap_ratio=MIN_OVERLAP_RATIO,
    n=N_TEST,
)

pd.set_option('display.float_format', lambda v: f'{v:.3f}')
summary_row = pd.DataFrame([{'config': 'baseline_v2 (cascade)', **summ_v2.to_dict()}])
summary_row

## 4. Sauvegarde + comparaison vs baseline précédent

On persiste le CSV + summary JSON à côté des anciens runs (répertoire `outputs/msd_implementation/dino_medsam_cascade/metrics/`), avec un préfixe `baseline_v2_` pour qu'ils ne se mélangent pas.

In [ ]:
from datetime import datetime
save_dir = output_dir("msd_implementation", "dino_medsam_cascade", "metrics")
save_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime('%Y-%m-%dT%H-%M-%S')
csv_path = save_dir / f'baseline_v2_{ts}.csv'
json_path = save_dir / f'baseline_v2_{ts}_summary.json'
df_v2.to_csv(csv_path, index=False)
with open(json_path, 'w') as f:
    json.dump(summ_v2.to_dict(), f, indent=2)
print('saved', csv_path)
print('saved', json_path)

In [ ]:
# Comparaison au baseline précédent (valeurs du notebook improved_pipeline.ipynb)
prev_baseline = {
    'config': 'baseline_v1 (single prompt "tumor.")',
    'sensitivity': 1.00, 'specificity': 0.00,
    'precision': 0.5000, 'f1': 0.6667,
    'dice_tumor_present_mean': 0.4472, 'dice_all_mean': 0.2236,
    'tp': 50, 'fp': 50, 'tn': 0, 'fn': 0,
}
collab_ref = {
    'config': 'collaborator (test_msd_no_resnet.py)',
    'sensitivity': 0.88, 'specificity': 0.00,
    'precision': 0.4680, 'f1': 0.6107,
    'dice_tumor_present_mean': 0.60, 'dice_all_mean': float('nan'),
    'tp': 44, 'fp': 50, 'tn': 0, 'fn': 6,
}
v2_row = {'config': 'baseline_v2 (cascade)', **summ_v2.to_dict()}
cmp = pd.DataFrame([prev_baseline, collab_ref, v2_row])
cmp[['config', 'sensitivity', 'specificity', 'precision', 'f1',
     'dice_tumor_present_mean', 'dice_all_mean', 'tp', 'fp', 'tn', 'fn']]

## 5. Diagnostic — scores DINO par catégorie (tumor-présent vs absent)

Pour préparer la recalibration du seuil (étape P4 du plan), on regarde la distribution des scores DINO sur tumeur-présent vs tumeur-absent.

In [ ]:
stats = df_v2.groupby('has_tumor')['score'].describe()[['count', 'mean', 'min', '50%', 'max']]
print(stats.to_string())

df_no_tumor = df_v2[~df_v2['has_tumor']]
df_tumor = df_v2[df_v2['has_tumor']]
if not df_no_tumor.empty and not df_tumor.empty:
    max_fp_score = df_no_tumor['score'].max()
    sim_fp = int((df_no_tumor['score'] > max_fp_score).sum())
    sim_fn = int((df_tumor['score'] <= max_fp_score).sum())
    print(f"\nScore max sur scan sain  : {max_fp_score:.4f}")
    print(f"Si seuil = {max_fp_score:.4f} → FP restants {sim_fp}/{len(df_no_tumor)}, FN ajoutés {sim_fn}/{len(df_tumor)}")

df_fn = df_tumor[df_tumor['dice'] == 0.0]
print(f"\nTumeurs totalement ratées (Dice=0) : {len(df_fn)}/{len(df_tumor)}")

## 6. Padding de box + post-process masque

Trois leviers, chacun modifiable séparément:

- **1** — *parity check* du preprocessing MedSAM vs `test_msd_no_resnet.py`
- **2** — padding `pad_frac` de la box tumor avant MedSAM (0.05 à 0.12 typique)
- **3** — post-process : `keep_largest` (composante connexe max), `min_area_px` / `max_area_frac` (filtre de taille)

Le masque MedSAM est déjà lissé par `medsam.postprocess_mask` (closing + drop des petites comp. < 5 %). 3 ajoute un filtre *plus strict*.

### 6.1 Parity preprocessing MedSAM

*   Élément de liste
*   Élément de liste



On compare au flot de `test_msd_no_resnet.py:157-167`. Les deux font exactement :

```python
img_1024 = skimage.transform.resize(img, (1024,1024), order=3, preserve_range=True, anti_aliasing=True).astype(uint8)
img_1024 = (img_1024 - img_1024.min()) / (img_1024.max() - img_1024.min())   # min-max sur 1024x1024
box_1024 = box / [W,H,W,H] * 1024
```

Différences :
- On passe par `medsam.encode_image` (bf16 + cache)
- `medsam.segment` applique `postprocess_mask` (closing k=2, drop components < 5% du plus grand). Dans la pratique ça aide le Dice sans toucher la détection.

In [ ]:
# Audit visuel montre qu'avec pad_frac=0 et pas de post-process supplémentaire,

# on retombe bien sur le chiffre baseline_v2 précédent (~0.576).

df_parity, summ_parity = eval_v2.run_baseline_v2(

    coco_json=config.MSD_TEST_JSON,

    text=TEXT_PROMPT, pancreas_thr=PANCREAS_THR, tumor_thr=TUMOR_THR,

    pancreas_margin_px=PANCREAS_MARGIN_PX, min_overlap_ratio=MIN_OVERLAP_RATIO,

    pad_frac=0.0, keep_largest=False,

    n=N_TEST, progress_desc='parity (pad=0, no-pp)',

)

print(f"Dice tumor-present : {summ_parity.dice_tumor_present_mean:.4f}  (attendu ~{summ_v2.dice_tumor_present_mean:.4f})")

### 6.2 Ablation P2 — grille padding × post-process

Cinq configs ciblées (on évite la grille complète pour rester sous ~50 min GPU) :

| Config | pad_frac | keep_largest | min_area_px | max_area_frac |
|---|---|---|---|---|
| A — baseline v2 | 0.00 | no | 0 | 1.0 |
| B — pad 5%       | 0.05 | no | 0 | 1.0 |
| C — pad 8%       | 0.08 | no | 0 | 1.0 |
| D — pad 8% + CC max | 0.08 | yes | 0 | 1.0 |
| E — pad 8% + CC + size filter | 0.08 | yes | 50 | 0.20 |

In [ ]:
p2_configs = [

    dict(name='A baseline v2',            pad_frac=0.00, keep_largest=False, min_area_px=0,  max_area_frac=1.0),

    dict(name='B pad 5%',                 pad_frac=0.05, keep_largest=False, min_area_px=0,  max_area_frac=1.0),

    dict(name='C pad 8%',                 pad_frac=0.08, keep_largest=False, min_area_px=0,  max_area_frac=1.0),

    dict(name='D pad 8% + CC max',        pad_frac=0.08, keep_largest=True,  min_area_px=0,  max_area_frac=1.0),

    dict(name='E pad 8% + CC + size',     pad_frac=0.08, keep_largest=True,  min_area_px=50, max_area_frac=0.20),

]



p2_results = {}

for cfg in p2_configs:

    name = cfg.pop('name')

    print(f'\n=== {name} ===')

    df_c, summ_c = eval_v2.run_baseline_v2(

        coco_json=config.MSD_TEST_JSON,

        text=TEXT_PROMPT, pancreas_thr=PANCREAS_THR, tumor_thr=TUMOR_THR,

        pancreas_margin_px=PANCREAS_MARGIN_PX, min_overlap_ratio=MIN_OVERLAP_RATIO,

        n=N_TEST, progress_desc=name,

        **cfg,

    )

    p2_results[name] = (df_c, summ_c)

    cfg['name'] = name  # restore for later

In [ ]:
rows = []

for name, (_, summ) in p2_results.items():

    rows.append({'config': name, **summ.to_dict()})

p2_df = pd.DataFrame(rows)[['config', 'sensitivity', 'specificity', 'precision', 'f1',

                             'dice_tumor_present_mean', 'dice_all_mean', 'tp', 'fp', 'tn', 'fn']]

p2_df

In [ ]:
# Persister chaque run + un summary global

p2_ts = datetime.now().strftime('%Y-%m-%dT%H-%M-%S')

summary_out = {}

for name, (df_c, summ_c) in p2_results.items():

    safe = name.replace(' ', '_').replace('%', 'pct').replace('+', 'p')

    df_c.to_csv(save_dir / f'p2_{p2_ts}_{safe}.csv', index=False)

    summary_out[name] = summ_c.to_dict()

with open(save_dir / f'p2_{p2_ts}_summary.json', 'w') as f:

    json.dump(summary_out, f, indent=2)

print('saved p2 ablation to', save_dir)